In [1]:
import pandas as pd
import torch
import torch.nn as nn
import sys
sys.path.append('../')
from utilities import load_embedding

In [2]:
class HAEncoder(nn.Module):
    def __init__(self, model: nn.Module):
        super().__init__()
        # 整个模型作为 submodule 挂上来
        self.hana = model

    def forward(self, ha_matrix, attention_mask=None):
        """
        ha_matrix: 你的 HA 输入张量，形状要和原来进 matrix_pooler 前一样
        attention_mask: 如果 matrix_pooler 需要，就传；不需要就可以省略
        """
        x = self.hana.matrix_dropout(ha_matrix)
        x = self.hana.matrix_pooler(x, attention_mask)  # 还是 2560 维
        
        # self.hana.linear 是 ModuleList -> 里面第0个又是 ModuleList(Linear, GELU)
        x = self.hana.linear[0][0](x)  # Linear: 2560 -> 256
        x = self.hana.linear[0][1](x)  # GELU

        return x   # [batch, ..., 256] 就是你要的 256 维向量


In [3]:
train_data = pd.read_csv('../../data/data_40/titer/train.csv')
device = torch.device('cuda:2')
model = torch.load('../../trained_model/1.7_Artificial_back/2025-08-19_17-43-32.pth', weights_only=False, map_location=device)
ha_encoder = HAEncoder(model)

In [4]:
# load embedding
emb_IDs = train_data['seq_id_d'].unique().tolist()
emb_files = ['matrix_' + ID + '.pt' for ID in emb_IDs]
emb_dict_active = load_embedding("../../data/data_40/embedding_Crick", files=emb_files)

Loading tensor: 100%|██████████| 4391/4391 [11:58<00:00,  6.11file/s]


In [5]:
from tqdm.auto import tqdm  # 建议用 auto，notebook/终端都兼容

vecs = []
model.to('cpu')

for matrix in tqdm(emb_dict_active.values(),
                   total=len(emb_dict_active),
                   desc="Encoding HA"):
    matrix = matrix.unsqueeze(0)  # (1, L, ...)
    mask = torch.ones(1, matrix.shape[1])  # 如果之前就是 float/CPU，就保持一致
    with torch.no_grad():  # 通常不需要梯度
        vec = ha_encoder(matrix, mask)
    vecs.append(vec)


Encoding HA:   0%|          | 0/4391 [00:00<?, ?it/s]

In [6]:
t_cat = torch.cat(vecs, dim=0)
np_arr = t_cat.detach().cpu().numpy()

In [8]:
import os
import torch
from tqdm.auto import tqdm

save_dir = "../../data/data_40/vector_Crick"
os.makedirs(save_dir, exist_ok=True)

# 取出字典的 keys，顺序与 values() 一致（Python 3.7+ 默认保持插入顺序）
keys = list(emb_dict_active.keys())

assert len(keys) == len(vecs), "keys 数量和 vecs 长度不一致，请检查！"

for key, vec in tqdm(zip(keys, vecs),
                     total=len(keys),
                     desc="Saving vecs"):
    vec_to_save = vec.detach().cpu()
    save_path = os.path.join(save_dir, f"{key}.pth")
    torch.save(vec_to_save, save_path)


Saving vecs:   0%|          | 0/4391 [00:00<?, ?it/s]